# Examen Parcial

**Grupo:** Tomas Vargas, Miguel Hernandez y Jossemar Lopoez

In [1]:
import pandas as pd
URL = ("https://storage.googleapis.com/emb-prod-bkt-publicdata/"
"public-downloads/monthly_full_release_long_format.csv")
cols = ["Area", "Date", "Variable", "Unit", "Value"]
crudo = pd.read_csv(URL, usecols=cols, parse_dates=["Date"])
cr = crudo[(crudo["Area"] == "Costa Rica") & (crudo["Unit"] == "TWh")]
cr = (cr.pivot_table(index="Date", columns="Variable", values="Value")
* 1000).asfreq("MS")[:"2026-06"]
hidro = cr["Hydro"].rename("hidro")
fosil = cr["Fossil"].rename("fosil")
print(hidro.shape)

(90,)


> **Recordatorios**
> - El notebook debe correr en orden y sin errores; **cada celda de código debe producir una salida visible** (gráfico o `print`).
> - Entregar `Parcial1_Grupo3.ipynb` + `predicciones_hidro.csv` antes de las 4:00 p. m. del jueves 5/11/2026.
> - Partición fija: entrenamiento ene-2019 a jun-2025 (78 meses), prueba jul-2025 a jun-2026 (12 meses).
> - Métricas siempre: RMSE, MAE y MASE (m = 12, escalado con el ingenuo estacional del entrenamiento).

# Parte 1: Aplicación en Python (60 pts)

### Preparación: librerías, partición y funciones de métricas

In [ ]:
# TODO: imports (numpy, matplotlib, statsmodels, etc.)

# TODO: partición train (2019-01 a 2025-06) y test (2025-07 a 2026-06)
# train = ...
# test = ...

# TODO: funciones rmse(), mae(), mase(y_true, y_pred, y_train, m=12)

# print(train.shape, test.shape)  # esperado: (78,) (12,)

## 1.a. Exploración (10 pts)

**(I)** Graficar la serie completa de generación hidroeléctrica y, en otro gráfico, la generación fósil.

In [ ]:
# TODO: gráfico de hidro

In [ ]:
# TODO: gráfico de fósil

**(II)** Describir en dos o tres frases la tendencia y la estacionalidad. ¿Qué meses generan más y cuáles menos? ¿Qué relación hay con la época seca y la lluviosa?

In [ ]:
# TODO (opcional): promedio por mes del año para apoyar la respuesta

**Respuesta:**

_(escribir aquí)_

**(III)** Descomponer la serie (`seasonal_decompose`, periodo 12) y comentar qué pasó en 2023 y 2024, apoyándose en el gráfico de generación fósil.

In [ ]:
# TODO: seasonal_decompose(hidro, period=12) y graficar

**Respuesta:**

_(escribir aquí)_

## 1.b. Un modelo clásico (12 pts)

**(I)** Ajustar Holt-Winters o SARIMA (statsmodels) sobre el entrenamiento. Justificar la configuración: ¿tendencia sí o no?, ¿qué órdenes?, ¿constante?

In [ ]:
# TODO: ajuste del modelo clásico sobre train

**Justificación de la configuración:**

_(escribir aquí)_

**(II)** Pronosticar los 12 meses de prueba, calcular RMSE, MAE y MASE, y graficar observado vs. pronosticado.

In [ ]:
# TODO: pronóstico de 12 meses + métricas

In [ ]:
# TODO: gráfico observado vs. pronosticado

## 1.c. Un modelo moderno (12 pts)

**(I)** Con la misma partición, ajustar Prophet o usar Chronos-2 (zero-shot). Instalar en Colab.

In [ ]:
# TODO: !pip install prophet   (o chronos-forecasting)

In [ ]:
# TODO: ajuste / pronóstico del modelo moderno sobre la misma partición

**(II)** Calcular las mismas tres métricas y graficar.

In [ ]:
# TODO: métricas RMSE, MAE, MASE

In [ ]:
# TODO: gráfico observado vs. pronosticado

## 1.d. Comparación y validación (14 pts)

**(I)** Tabla con RMSE, MAE y MASE de los dos modelos y del ingenuo estacional.

In [ ]:
# TODO: pronóstico del ingenuo estacional + DataFrame comparativo

**(II)** Ventana expansiva con al menos tres inicios de prueba (jul-2023, jul-2024, jul-2025; 12 meses cada uno). Reportar el MASE de cada modelo en cada ventana y su promedio.

In [ ]:
# TODO: loop sobre inicios ['2023-07', '2024-07', '2025-07']
#   - train = todo lo anterior al inicio; test = 12 meses desde el inicio
#   - reajustar cada modelo, pronosticar, calcular MASE (escalado con ESE train)
# TODO: tabla MASE por ventana + promedio

**(III)** Elegir el mejor modelo con un criterio explícito (p. ej. MASE promedio y estabilidad entre ventanas) y justificar en 3–5 frases. Comentar qué le pasó a cada modelo en la ventana que incluye 2024.

**Criterio y elección:**

_(escribir aquí)_

**Qué pasó en la ventana con 2024:**

_(escribir aquí)_

## 1.e. Pronóstico final y riesgo (12 pts)

**(I)** Reentrenar el modelo elegido con toda la serie (ene-2019 a jun-2026) y pronosticar jul-2026 a jun-2027 con banda del 80 %.

In [ ]:
# TODO: reentrenar con hidro completa + pronóstico 12 meses + banda 80 % (p10, p90)

In [ ]:
# TODO: gráfico de la serie + pronóstico + banda

**(II)** Guardar las predicciones en `predicciones_hidro.csv` con columnas `mes`, `pronostico`, `p10`, `p90`.

In [ ]:
# TODO: armar DataFrame y .to_csv('predicciones_hidro.csv', index=False); imprimirlo

**(III)** Con Monte Carlo (`simulate` de statsmodels o remuestreo de residuos), estimar la probabilidad de que la generación de marzo de 2027 sea menor que 550 GWh.

In [ ]:
# TODO: N simulaciones de trayectorias, tomar marzo-2027, estimar P(valor < 550)

**(IV)** Dos o tres frases para una persona que planifica el sistema eléctrico: ¿qué le dirían y con cuánta confianza?

**Respuesta:**

_(escribir aquí)_

# Parte 2: Interpretación conceptual y de código (25 pts)

_Respuestas en celdas de texto; no requieren código._

## 2.1. Regularización (6 pts)

**(a)** ¿Qué hacen Ridge y Lasso sobre los coeficientes y por qué ayudan a controlar el sobreajuste?

**Respuesta:**

_(escribir aquí)_

**(b)** Una diferencia práctica entre Ridge y Lasso, relacionada con el compromiso sesgo–varianza.

**Respuesta:**

_(escribir aquí)_

**(c)** ¿Cómo es posible que Ridge y Lasso dieran mayor RMSE de prueba que mínimos cuadrados? ¿Por qué se validó con `TimeSeriesSplit` y no con K-fold común?

**Respuesta:**

_(escribir aquí)_

## 2.2. Cadena de Markov con datos reales (7 pts)

$$P = \begin{pmatrix} 0.83 & 0.17 \\ 0.29 & 0.71 \end{pmatrix}$$

(A: hidro ≥ 700 GWh, B: hidro baja; filas = mes actual, columnas = mes siguiente)

**(a)** Interpretar en palabras la segunda fila.

**Respuesta:**

_(escribir aquí)_

**(b)** ¿Qué representa la distribución estacionaria? Plantear $\pi P = \pi$ con $\pi_A + \pi_B = 1$ y resolver.

**Respuesta:**

_(escribir aquí)_

**(c)** ¿Es razonable el supuesto de Markov para la hidrología de un país con años de El Niño? Argumentar.

**Respuesta:**

_(escribir aquí)_

**(d)** En una frase, conectar la cadena de Markov con PageRank o con los modelos de lenguaje.

**Respuesta:**

_(escribir aquí)_

## 2.3. Leer código (6 pts)

```python
import numpy as np
rng = np.random.default_rng(2026)
def log_post(p):  # 18 exitos en 50 intentos, prior plano
    return 18*np.log(p) + 32*np.log(1-p) if 0 < p < 1 else -np.inf
p, cadena = 0.5, []
for _ in range(20000):
    prop = p + rng.normal(0, 0.1)
    if np.log(rng.uniform()) < log_post(prop) - log_post(p):
        p = prop
    cadena.append(p)
print(np.mean(cadena[2000:]))
```

**(a)** ¿Monte Carlo o MCMC? ¿Qué distribución muestrea y qué valor aproximado imprime? Justificar.

**Respuesta:**

_(escribir aquí)_

**(b)** Papel de `prop`, de la comparación con `rng.uniform()` y del descarte de las primeras 2000 iteraciones.

**Respuesta:**

_(escribir aquí)_

**(c)** En dos frases, diferencia entre la simulación Monte Carlo de 1.e(III) y este código.

**Respuesta:**

_(escribir aquí)_

## 2.4. Leer resultados (6 pts)

**(a)** MASE promedio 1.10 (mejor modelo) vs. 1.40 (ingenuo estacional): ¿es malo el modelo por tener MASE > 1? ¿Qué compara exactamente el MASE?

**Respuesta:**

_(escribir aquí)_

**(b)** ¿Qué significa que Chronos-2 sea zero-shot? Una situación donde es ventaja y otra donde no basta.

**Respuesta:**

_(escribir aquí)_

**(c)** Con un año de prueba ganaba SARIMA; con tres ventanas, Holt-Winters. ¿Qué enseña sobre cómo elegir un modelo?

**Respuesta:**

_(escribir aquí)_

# Parte 3: Ejercicios a mano (15 pts)

_Resolver a mano con todo el procedimiento e insertar el escaneo (imagen o PDF)._

## 3.1. Crecimiento exponencial (5 pts)

$\frac{dD}{dt} = rD$, $D(2020) = 10\,000$ GWh, $D(2022) = 11\,025$ GWh.

(a) Resolver y encontrar $r$ (exacto y aproximado). (b) Pronosticar 2024.

_Insertar imagen del desarrollo:_ `![3.X](ruta_o_url_de_la_imagen.png)`

## 3.2. Métricas de error (5 pts)

| | Mes 1 | Mes 2 | Mes 3 | Mes 4 |
|---|---|---|---|---|
| Real | 700 | 620 | 580 | 760 |
| Pronóstico | 680 | 650 | 560 | 800 |

(a) MAE y RMSE. (b) MASE con MAE del ingenuo estacional = 80 GWh, e interpretar. (c) ¿Por qué RMSE > MAE? ¿Qué error pesa más?

_Insertar imagen del desarrollo:_ `![3.X](ruta_o_url_de_la_imagen.png)`

## 3.3. Cadena de dos estados (5 pts)

$P = \begin{pmatrix} 0.9 & 0.1 \\ 0.3 & 0.7 \end{pmatrix}$, estados A y B.

(a) Si hoy está en A, distribución dentro de dos meses. (b) Distribución estacionaria.

_Insertar imagen del desarrollo:_ `![3.X](ruta_o_url_de_la_imagen.png)`

# Declaración de uso de inteligencia artificial

_Herramientas usadas y para qué:_

- _(completar)_